# Train YOLOv8n cho Drone trên Google Colab

Hướng dẫn từng bước để train mô hình YOLOv8n (nano) trên Google Colab, sau đó xuất ra định dạng để chạy trên drone.

**Bước đầu tiên:** vào menu `Runtime > Change runtime type > Hardware accelerator > T4 GPU > Save`, rồi chạy lần lượt từng ô từ trên xuống.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print(">>> Chưa có GPU. Vào menu: Runtime > Change runtime type > chọn 'T4 GPU' rồi chạy lại.")

In [ ]:
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Tải dataset từ Roboflow

Dataset đã được export sẵn sang định dạng YOLOv8, kèm file `data.yaml` (chứa tên class). Ô dưới đây sẽ tải về và giải nén vào `/content/`.

In [ ]:
import osfrom roboflow import Roboflow# ===== THÔNG TIN ROBOFLOW CỦA BẠN =====# Nạp key từ file .env (chạy local). Trên Kaggle/Colab: đặt biến môi trường ROBOFLOW_API_KEY.def _load_env():    try:        for line in open(".env"):            line = line.strip()            if line and not line.startswith("#") and "=" in line:                k, v = line.split("=", 1)                os.environ.setdefault(k.strip(), v.strip())    except FileNotFoundError:        pass_load_env()ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")if not ROBOFLOW_API_KEY:    raise ValueError("Thiếu ROBOFLOW_API_KEY. Tạo file .env (từ .env.example) và điền key, hoặc set biến môi trường.")WORKSPACE         = "trantungbach26-gmail-com"PROJECT_NAME      = "citrus-disease-detection-yoydc-ahtka"PROJECT_VERSION   = 1rf = Roboflow(api_key=ROBOFLOW_API_KEY)project = rf.workspace(WORKSPACE).project(PROJECT_NAME)version = project.version(PROJECT_VERSION)version.download("yolov8")

In [ ]:
# Tự động tìm thư mục dataset vừa tải (thư mục chứa data.yaml)
import os, glob

candidates = glob.glob("/content/*/data.yaml")
if candidates:
    DATASET_PATH = os.path.dirname(candidates[0])
else:
    # Nếu không tìm thấy, sửa lại đường dẫn tên thư mục cho đúng
    DATASET_PATH = "/content/citrus-disease-detection-1"

print("DATASET_PATH =", DATASET_PATH)
print("Cấu trúc thư mục:")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)
    print("  " * level + "|" + os.path.basename(root) + "/")
    for f in files[:3]:
        print("  " * (level + 1) + f)

import yaml
with open(os.path.join(DATASET_PATH, "data.yaml")) as f:
    cfg = yaml.safe_load(f)
print("\nSố class:", cfg["nc"])
print("Tên class:", cfg["names"])

## 2. Load model & train

File `yolov8n.pt` (mô hình nano) sẽ được tải tự động lần đầu tiên.

In [ ]:
import os

if os.path.exists("/content/yolov8n.pt"):
    model_path = "/content/yolov8n.pt"
elif os.path.exists("/content/drive/MyDrive/yolov8n.pt"):
    model_path = "/content/drive/MyDrive/yolov8n.pt"
else:
    model_path = "yolov8n.pt"  # ultralytics tự tải về

print("Dùng model:", model_path)
model = YOLO(model_path)

In [ ]:
# ===== Cấu hình train (best practice cho Colab T4 miễn phí + yolov8n) =====
# - epochs=50 là đủ: yolov8n hội tụ nhanh, 100 epochs trên T4 mất ~8.5h -> bị cut giữa chừng
# - patience=20: nếu val không cải thiện 20 epochs -> tự dừng sớm (thường dừng quanh 30-40)
EPOCHS = 50       # số vòng lặp tối đa
IMGSZ = 640       # kích thước huấn luyện. Export/kmodel sẽ dùng CHÍNH con số này
BATCH = 16        # T4 GPU chạy tốt 16-32 với yolov8n
PATIENCE = 20     # dừng sớm nếu val không cải thiện sau N epochs

# Nơi ultralytics lưu kết quả (tên "drone_yolov8n" phải khớp name= ở dưới)
RESULTS_DIR = "/content/runs/drone_yolov8n/weights"

# ===== AUTO-BACKUP: cứ mỗi epoch xong, tự copy best.pt lên Google Drive =====
# Nếu Colab bị thu hồi giữa chừng, bạn vẫn giữ được bản tốt nhất tới lúc đó.
import os, shutil
from ultralytics.utils import callbacks

def _backup_to_drive(trainer):
    try:
        src = os.path.join(trainer.save_dir, "weights", "best.pt")
        dst = "/content/drive/MyDrive/drone_yolo/best_checkpoint.pt"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
        print(f"  [backup epoch {trainer.epoch}] best.pt -> Drive", flush=True)
    except Exception as e:
        print("  [backup fail]", e, flush=True)

callbacks.default_callbacks["on_fit_epoch_end"].append(_backup_to_drive)

print(f"Train yolov8n: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, patience={PATIENCE}")

results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    seed=42,
    cache=True,      # cache ảnh vào RAM để train nhanh hơn
    workers=2,       # an toàn trên Colab, tránh treo
    cos_lr=True,     # giảm learning rate theo cos, hội tụ tốt hơn
    project="/content/runs",
    name="drone_yolov8n",
)

## 3. Đánh giá kết quả

In [ ]:
# Đánh giá trên tập validation
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision:{metrics.box.mp:.4f}")
print(f"Recall:   {metrics.box.mr:.4f}")

In [ ]:
import glob

# Dự đoán thử trên ảnh test (Roboflow dùng thư mục valid/test thay vì val)
test_imgs = (glob.glob(f"{DATASET_PATH}/test/images/*.*")
             or glob.glob(f"{DATASET_PATH}/valid/images/*.*")
             or glob.glob(f"{DATASET_PATH}/val/images/*.*"))
print("Tìm thấy", len(test_imgs), "ảnh")

if test_imgs:
    model.predict(source=test_imgs[:8], conf=0.25, save=True)
    # Ảnh kết quả nằm trong thư mục runs/detect/predict
    from IPython.display import Image
    display(Image(filename="/content/runs/detect/predict/" + os.path.basename(test_imgs[0])))

## 4. Xuất model để chạy trên drone

Chọn định dạng phù hợp với phần cứng bạn dùng để bay (điều khiển trên bo mạch):
- **Jetson Nano / Jetson Orin (NVIDIA)**: `tensorrt` hoặc `onnx`
- **Raspberry Pi / board nhúng**: `ncnn` hoặc `onnx`
- **Điện thoại Android / Edge TPU**: `tflite`
- **Máy tính Intel**: `openvino`

In [ ]:
# ===== EXPORT: 1 lần train -> 2 phiên bản =====
# [LAPTOP test]  best.pt   -> chạy trực tiếp bằng ultralytics trên máy bạn
# [DRONE K230]   best.onnx -> qua nncase -> best.kmodel để chạy trên board

# (1) ONNX dành cho K230 — KHÔNG dùng half=True để nncase nạp được
#     imgsz=IMGSZ phải KHỚP với model_input_size trong k230_yolov8_det.py
model.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)

# (2) Bản int8 lượng tử, nhẹ hơn (tùy chọn)
model.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True, int8=True, data=f"{DATASET_PATH}/data.yaml")

print("Export xong. File trong:", RESULTS_DIR)
print("  - best.pt   -> LAPTOP (chạy realtime bằng ultralytics)")
print("  - best.onnx -> DRONE K230 (convert_to_kmodel.py -> best.kmodel)")

In [ ]:
# Lưu mọi phiên bản model về Google Drive để không mất khi Colab đóng session.
import shutil, glob

SAVE_DIR = "/content/drive/MyDrive/drone_yolo"
os.makedirs(SAVE_DIR, exist_ok=True)

# best.pt + last.pt + mọi .onnx / .tflite vừa export
files = glob.glob(f"{RESULTS_DIR}/best.*") + glob.glob(f"{RESULTS_DIR}/last.*")

for src in files:
    shutil.copy(src, os.path.join(SAVE_DIR, os.path.basename(src)))
    print("Đã lưu:", os.path.basename(src))

print("Thư mục Drive:", SAVE_DIR)

## Mẹo khi train cho drone

- **Vật thể nhỏ**: ảnh chụp từ drone thường có vật thể rất nhỏ. Thử tăng `imgsz` lên 800 hoặc dùng kỹ thuật tile/split ảnh.
- **Không lật dọc (flipud)**: với ảnh trên cao, giữ nguyên default `flipud=0.0` để tránh vật thể lộn ngược không đúng thực tế. `fliplr=0.5` (lật ngang) vẫn nên giữ.
- **Augmentation**: `mosaic` mặc định là 1.0, rất hữu ích cho vật thể nhỏ.
- **Dừng sớm**: nếu thấy `best.pt` không cải thiện ở nhiều epoch, giảm `patience` để đỡ tốn thời gian.
- **Session mất kết nối**: Colab miễn phí có thể ngắt sau vài giờ. Nên đã mount Drive từ đầu để `best.pt` được lưu lại, hoặc dùng Colab Pro để train lâu hơn.